# Semáforo inteligente de estrés académico: entrenamiento y conversión a TFLite (int8)
**Proyecto Integrador · Ingeniería de Datos e IA · Universidad Santo Tomás, Tunja**

Este notebook cubre:
1. Cargar `dataset_v1.csv`, que se genera con `preparar_dataset.py`.
2. Entrenar una red neuronal pequeña con 3 clases: 0 = verde, 1 = amarillo, 2 = rojo.
3. Evaluarla con accuracy, F1 y matriz de confusión sobre personas que no vio en el entrenamiento.
4. Convertirla a **TensorFlow Lite int8**, que corresponde a la conversión documentada de R3.
5. Generar `modelo_estres.h` y `escalado.h` para cargarlos en el ESP32 (R2).

> Ejecuta las celdas en orden con **Entorno de ejecución → Ejecutar todas**.

## 1. Librerías

In [ ]:
import numpy as np, pandas as pd, tensorflow as tf, matplotlib.pyplot as plt, os
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight
tf.random.set_seed(42); np.random.seed(42)
print("TensorFlow", tf.__version__)

## 2. Cargar el dataset
Se lee `dataset_v1.csv` desde el repositorio de GitHub (o se sube a mano). Si todavía no lo tienen, pon `MODO_DEMO = True` para probar que todo funcione con datos sintéticos.
**Las métricas en modo demo NO se pueden reportar en el artículo.**

In [ ]:
MODO_DEMO = False   # True solo para probar el flujo sin datos reales
CARACT = ["bpm_prom", "bpm_std", "temp_prom", "temp_pend", "mov_prom", "mov_std"]
CLASES = ["Verde", "Amarillo", "Rojo"]

if MODO_DEMO:
    rng = np.random.default_rng(0); filas = []
    for p in range(15):
        for e, (bpm, temp, mov) in enumerate([(72, 33.5, .05), (84, 33.8, .12), (96, 34.1, .20)]):
            for _ in range(15):
                filas.append(dict(participante=f"P{p:02d}", etiqueta=e,
                    bpm_prom=rng.normal(bpm, 6), bpm_std=abs(rng.normal(2 + e, 1)),
                    temp_prom=rng.normal(temp, .4), temp_pend=rng.normal(.02 * e, .03),
                    mov_prom=abs(rng.normal(mov, .05)), mov_std=abs(rng.normal(.02 + .01 * e, .01))))
    ds = pd.DataFrame(filas)
    per = sorted(ds.participante.unique())
    ds["particion"] = ds.participante.map({p: "train" if i < 10 else ("val" if i < 12 else "test") for i, p in enumerate(per)})
else:
    DATA_URL = "https://raw.githubusercontent.com/sarahcubides12-ctrl/semaforo-estres/main/dataset/dataset_v1.csv"
    try:
        ds = pd.read_csv(DATA_URL)          # lee el dataset directo del repositorio de GitHub
        print("Dataset leído de GitHub")
    except Exception:
        from google.colab import files     # si no hay internet o el repo no lo tiene, se sube a mano
        subido = files.upload()
        ds = pd.read_csv(next(iter(subido)))

print(ds.shape); print(pd.crosstab(ds["particion"], ds["etiqueta"], margins=True))

## 3. Separar en train, val y test (por persona) y escalar

In [ ]:
tr, va, te = (ds[ds.particion == s] for s in ["train", "val", "test"])
Xtr, Xva, Xte = (d[CARACT].values.astype("float32") for d in (tr, va, te))
ytr, yva, yte = (d["etiqueta"].values.astype("int32") for d in (tr, va, te))

# Escalado z-score con estadísticas SOLO de train (se copian al ESP32)
media = Xtr.mean(axis=0); desv = Xtr.std(axis=0) + 1e-6
esc = lambda X: (X - media) / desv
Xtr_s, Xva_s, Xte_s = esc(Xtr), esc(Xva), esc(Xte)

pesos = compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)
pesos = dict(enumerate(pesos))
print("Train:", len(ytr), "| Val:", len(yva), "| Test:", len(yte), "| Pesos:", pesos)

## 4. Modelo: red densa pequeña, pensada para el ESP32

In [ ]:
modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(CARACT),)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax"),
])
modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])
modelo.summary()

hist = modelo.fit(Xtr_s, ytr, validation_data=(Xva_s, yva), epochs=300, batch_size=16,
                  class_weight=pesos, verbose=0,
                  callbacks=[tf.keras.callbacks.EarlyStopping(patience=30, restore_best_weights=True)])
print("Épocas entrenadas:", len(hist.history["loss"]))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for k, a in zip(["loss", "accuracy"], ax):
    a.plot(hist.history[k], label="train"); a.plot(hist.history["val_" + k], label="val")
    a.set_title(k); a.set_xlabel("época"); a.legend()
plt.tight_layout(); plt.savefig("curvas_entrenamiento.png", dpi=150); plt.show()

## 5. Evaluación en test (personas nunca vistas)

In [ ]:
pred = modelo.predict(Xte_s, verbose=0).argmax(1)
acc_f = accuracy_score(yte, pred); f1_f = f1_score(yte, pred, average="macro")
print(f"Accuracy (float32): {acc_f:.3f} | F1 macro: {f1_f:.3f}\n")
print(classification_report(yte, pred, labels=[0, 1, 2], target_names=CLASES, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(yte, pred, labels=[0, 1, 2]), display_labels=CLASES).plot(cmap="Blues")
plt.title("Matriz de confusión (test)"); plt.savefig("matriz_confusion.png", dpi=150); plt.show()

## 6. Conversión a TensorFlow Lite
Se generan dos versiones:
- **float32**: la referencia.
- **int8 completo**: cuantización post-entrenamiento con un *representative dataset* tomado de train. Pesos, activaciones, entrada y salida quedan en int8, que es lo que se carga en el ESP32.

In [ ]:
# float32
tfl_f32 = tf.lite.TFLiteConverter.from_keras_model(modelo).convert()

# int8 completo
def rep_data():
    for x in Xtr_s[np.random.choice(len(Xtr_s), min(200, len(Xtr_s)), replace=False)]:
        yield [x.reshape(1, -1).astype("float32")]

conv = tf.lite.TFLiteConverter.from_keras_model(modelo)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep_data
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
tfl_i8 = conv.convert()

open("modelo_estres_f32.tflite", "wb").write(tfl_f32)
open("modelo_estres_int8.tflite", "wb").write(tfl_i8)
print(f"Tamaño float32: {len(tfl_f32)/1024:.2f} KB | int8: {len(tfl_i8)/1024:.2f} KB "
      f"(reducción {100*(1-len(tfl_i8)/len(tfl_f32)):.0f} %)")

## 7. Verificar el modelo int8 (debe dar casi lo mismo que float32)

In [ ]:
it = tf.lite.Interpreter(model_content=tfl_i8); it.allocate_tensors()
ent, sal = it.get_input_details()[0], it.get_output_details()[0]
s_in, z_in = ent["quantization"]; s_out, z_out = sal["quantization"]
print("Entrada  scale/zero_point:", s_in, z_in)
print("Salida   scale/zero_point:", s_out, z_out)

pred8 = []
for x in Xte_s:
    xq = np.clip(np.round(x / s_in + z_in), -128, 127).astype(np.int8).reshape(1, -1)
    it.set_tensor(ent["index"], xq); it.invoke()
    pred8.append(it.get_tensor(sal["index"])[0].argmax())
pred8 = np.array(pred8)
acc_8 = accuracy_score(yte, pred8); f1_8 = f1_score(yte, pred8, average="macro")
print(f"\nAccuracy int8: {acc_8:.3f} | F1 macro int8: {f1_8:.3f} | caída vs float32: {100*(acc_f-acc_8):.1f} pts")

## 8. Exportar para el ESP32: `modelo_estres.h` y `escalado.h`

In [ ]:
def a_header(datos, nombre):
    hexs = ", ".join(f"0x{b:02x}" for b in datos)
    return (f"// Generado automáticamente desde el notebook\n#pragma once\n"
            f"alignas(16) const unsigned char {nombre}[] = {{{hexs}}};\n"
            f"const unsigned int {nombre}_len = {len(datos)};\n")

open("modelo_estres.h", "w").write(a_header(tfl_i8, "modelo_estres"))

with open("escalado.h", "w") as f:
    f.write("// Escalado z-score (estadísticas de train) y cuantización de entrada/salida\n#pragma once\n")
    f.write("// Orden: " + ", ".join(CARACT) + "\n")
    f.write("const float MEDIA[6] = {" + ", ".join(f"{v:.6f}f" for v in media) + "};\n")
    f.write("const float DESV[6]  = {" + ", ".join(f"{v:.6f}f" for v in desv) + "};\n")
    f.write(f"const float IN_SCALE = {s_in:.8f}f;  const int IN_ZERO = {int(z_in)};\n")
    f.write(f"const float OUT_SCALE = {s_out:.8f}f; const int OUT_ZERO = {int(z_out)};\n")
print(open("escalado.h").read())

## 9. Resumen para el artículo y el informe de recursos (R3 / R7)

In [ ]:
resumen = pd.DataFrame({
    "Modelo": ["Keras float32", "TFLite float32", "TFLite int8"],
    "Tamaño (KB)": [None, round(len(tfl_f32)/1024, 2), round(len(tfl_i8)/1024, 2)],
    "Accuracy test": [round(acc_f, 3), round(acc_f, 3), round(acc_8, 3)],
    "F1 macro test": [round(f1_f, 3), round(f1_f, 3), round(f1_8, 3)],
})
print("Parámetros del modelo:", modelo.count_params())
if MODO_DEMO: print("\n⚠ MODO DEMO: estos números NO son reales, no los reporten.")
resumen

## 10. Copia de respaldo del modelo (texto base64)

In [ ]:
import base64
print("===TFLITE_B64_INICIO===")
print(base64.b64encode(tfl_i8).decode())
print("===TFLITE_B64_FIN===")
print("===ESCALADO_INICIO===")
print(open("escalado.h").read())
print("===ESCALADO_FIN===")

## 11. Descargar archivos

In [ ]:
from google.colab import files
for f in ["modelo_estres_int8.tflite", "modelo_estres.h", "escalado.h",
          "matriz_confusion.png", "curvas_entrenamiento.png"]:
    files.download(f)